# Data Engineering and Quality

Bronze/silver/gold artifacts, data contract checks, and missingness profile.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
from gtd_capstone.contracts import validate_data_contract
from gtd_capstone.data.cleaning import data_quality_report
from gtd_capstone.data.repository import DataRepository

repo = DataRepository()
incidents = repo.load_incidents(sample_rows=SAMPLE_ROWS)
quality = data_quality_report(incidents)
contract = validate_data_contract(incidents)
{
    "rows": quality["rows"],
    "columns": quality["columns"],
    "duplicate_eventids": quality["duplicate_eventids"],
    "contract_passed": contract["passed"],
}

{'rows': 25000,
 'columns': 37,
 'duplicate_eventids': 0,
 'contract_passed': False}

In [3]:
import pandas as pd

pd.DataFrame(quality["checks"])

,name,passed,detail
0,eventid_unique,True,eventid is the primary incident identifier.
1,year_range_present,True,Observed year range 1970-1985.
2,casualties_non_negative,True,Fatality and wounded columns are clipped at ze...
3,aggregate_geo_policy,True,API and frontend expose aggregated hotspots by...


In [4]:
missing = (
    incidents.isna()
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .rename("missing_rows")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing

,column,missing_rows
0,latitude,1234
1,longitude,1234
2,imonth,0
3,iyear,0
4,eventid,0
5,country_txt,0
6,iday,0
7,provstate,0
8,region_txt,0
9,city,0
